# DINO E3 — ablación del ancho de la MLP

**Estado: prospectivo; no ejecutado.** Objetivo: comprobar sensibilidad al ancho oculto. Pregunta: ¿512 unidades mejoran frente a 256? Hipótesis: un cuello de botella proporcionalmente mayor podría limitar adaptación; no se presupone mejora ni causalidad de pérdida de información. Intervención: solo ancho 256/512. Controles: YOLO y DINO congelados, corpus, pérdidas, tasas, sampler, presupuesto máximo y semillas iguales. Muestra: train 193.276 / val 273; test público 277 / PetTrace 502, separados. Seis entrenamientos (tres semillas por ancho); catorce evaluaciones incluyendo E2. Selección solo por validación. Métricas: Rank-1/5, mAP, EER descriptiva, media/DE e intervalos por identidad. Resultados: se generan abajo, nunca se anticipan. Límites: estudio retrospectivo, distinta capacidad y exposición efectiva; sin confirmación entre sesiones.

Métodos: [DINOv2](https://arxiv.org/abs/2304.07193), [batch-hard triplet](https://arxiv.org/abs/1703.07737), [YOLOv8](https://docs.ultralytics.com/models/yolov8/). La elección de anchos y el protocolo son decisiones del estudio. Ver `docs/dino_mlp_width_protocol.md`. No se modifica el experimento histórico ni se aplica LoRA.

## Fase 1. Preparación y verificación de datos

Cargar releases preparadas e inmutables. Configuración operativa previa: `tools/setup_dino_colab.py`; no instalaciones en este notebook. CUDA para extraer features si no hay caché. No sustituir el corpus por las fotos de test.

In [ ]:
import sys, os, json, gc, shutil, time
from pathlib import Path
from dataclasses import asdict
if not (3,11) <= sys.version_info[:2] < (3,14):
    raise RuntimeError('Python 3.11–3.13 required')
import numpy as np
import torch
import mlflow
from pettrace.tracking import PRODUCTION_URI, source_identity, publish_evaluation
from pettrace.data.verification import sha256
if not torch.cuda.is_available():raise RuntimeError('The full study requires CUDA')
for name in ('HF_TOKEN','MLFLOW_TRACKING_USERNAME','MLFLOW_TRACKING_PASSWORD'):
    if not os.environ.get(name):
        from google.colab import userdata
        os.environ[name]=userdata.get(name)
    if not os.environ[name]:raise RuntimeError('Missing '+name)
os.environ['MLFLOW_TRACKING_URI']=PRODUCTION_URI
os.environ['MLFLOW_REGISTRY_URI']=PRODUCTION_URI
mlflow.set_tracking_uri(PRODUCTION_URI);mlflow.set_registry_uri(PRODUCTION_URI)
LOG_TO_MLFLOW=True
# Registration is outside this ablation; no existing setting or alias is changed.
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
SEEDS=(45,46,47)
MODES=('mlp',)
WIDTHS=(256,512)
REPORT=Path('reports/dino-mlp-width/20261120-v1')
REPORT.mkdir(parents=True,exist_ok=True)
SOURCE=source_identity()
torch.set_num_threads(4)
from pettrace.data import training
from pettrace.experiments.e3 import load_benchmarks
from pettrace.evaluation.core import load_model
from pettrace.training.dinov2 import build, Config, Pipeline, Preprocessor, fit, REVISION, REPOSITORY
from pettrace.training import triplet, lora
from pettrace.experiments.dinov2 import frozen_features, publish_training
prepared=training(version='v2',cache_dir=str(REPORT/'data'))
assert prepared.fingerprint=='d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58'
rows=[r for r in prepared.rows if r['split'] in ('train','val')]
assert sum(r['split']=='train' for r in rows)==193276
assert sum(r['split']=='val' for r in rows)==273
benchmarks=load_benchmarks(prepared,REPORT,private_version='v6',private_commit=DATASET_COMMIT)
base=load_model('models:/pettrace-embedder/2');assert base.golden_ok
crop=lora.PreparedPreprocessor(base.pipeline,prepared)
gate=[next(r for r in rows if r['split']=='train' and r['species']==sp) for sp in ('cat','dog')]
crop.validate(gate)
study=dict(source=SOURCE,model_revision=REVISION,prepared=prepared.provenance,seeds=SEEDS,widths=WIDTHS,
    private_commit=DATASET_COMMIT,selection='width by mean validation macro mAP; seed by validation; frozen E2 eligible',
    training_budget='30 x 100 maximum; patience 7; P8K4',mlp_lr=1e-3,
    hypothesis='hidden width affects adaptation; transfer gain not assumed',
    public_test_commit='a724c1099a5432609c853958e88b68c4cf82d871',registration='outside this ablation')
# Reject reuse after code/configuration changes; use a new study directory instead.
study=json.loads(json.dumps(study))
if (REPORT/'study.json').exists():assert json.loads((REPORT/'study.json').read_text())==study
else:(REPORT/'study.json').write_text(json.dumps(study,indent=2))
print('Full data verified. Study:',REPORT)


## Fase 2. Representación y entrenamiento
Extraer una sola vez embeddings DINO+YOLO, compartidos por los seis ajustes. Solo la MLP cambia. Publicar cada entrenamiento obligatoriamente; reintentar publicación sin reentrenar. El mismo máximo de pasos no implica igual tiempo ni igual exposición tras parada temprana.

In [ ]:
from pettrace.evaluation.metrics import retrieval
from pettrace.models.golden import golden_images, compare, fingerprint
from huggingface_hub import snapshot_download
model_dir=Path(snapshot_download(REPOSITORY,revision=REVISION,allow_patterns=['model.safetensors','config.json','preprocessor_config.json']))
base_hashes={f:sha256(model_dir/f) for f in ('model.safetensors','config.json','preprocessor_config.json')}
base_hashes.update({'detector.'+k:v for k,v in base.sha256.items() if 'detector' in k})
encoder,processor=build('frozen','cuda')
pipeline=Pipeline(encoder,processor,base.pipeline.detector)
control_golden=pipeline.embed(golden_images());np.save(REPORT/'control-golden.npy',control_golden)
x=frozen_features(pipeline,rows,REPORT/'features',dict(model=base_hashes,processor=processor.to_dict(),source=SOURCE,detector_spec=base.pipeline.spec))
vi=[i for i,r in enumerate(rows) if r['split']=='val'];val=[rows[i] for i in vi]
control_metrics=retrieval(x[vi],[r['animal'] for r in val],[r['species'] for r in val])
control={k:float(np.mean([control_metrics[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
(REPORT/'control-validation.json').write_text(json.dumps(control_metrics,indent=2))
del encoder,pipeline;gc.collect();torch.cuda.empty_cache()
candidates=[]
for width in WIDTHS:
    for seed in SEEDS:
        folder=REPORT/f'mlp-h{width}-seed{seed}';folder.mkdir(exist_ok=True)
        config=triplet.Config(seed=seed,hidden=width)
        started=time.monotonic()
        if (folder/'training-result.json').exists():
            result=json.loads((folder/'training-result.json').read_text())
            assert result['config']==asdict(config)
        else:
            if (folder/'head.pt').exists():
                raise RuntimeError('Incomplete fit: preserve evidence and use a new study ID')
            _,result=triplet.fit(x,rows,folder,config,device='cpu',dimension=1024)
            (folder/'training-result.json').write_text(json.dumps(result,indent=2))
            (folder/'cost.json').write_text(json.dumps(dict(training_seconds=time.monotonic()-started,backend='cpu'),indent=2))
        checkpoint=folder/'head.pt';saved=torch.load(checkpoint,weights_only=True,map_location='cpu')
        assert saved['dimension']==1024 and saved['config']==asdict(config)
        head=triplet.Head(width,dimension=1024);head.load_state_dict(saved['state_dict'])
        _,values=triplet.validation_score(head,x[vi],val,'cpu')
        metrics={k:float(np.mean([values[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
        design=dict(study=study,mode=f'mlp-h{width}',seed=seed,config=asdict(config),checkpoint_sha256=sha256(checkpoint),trainable_parameters=lora.trainable_parameters(head),base_hashes=base_hashes)
        training_run=publish_training(folder,design,metrics)
        candidates.append(dict(mode='mlp',width=width,seed=seed,validation=metrics,folder=str(folder),training_run=training_run))
means={w:float(np.mean([c['validation']['mAP'] for c in candidates if c['width']==w])) for w in WIDTHS}
width=min(WIDTHS,key=lambda w:(-means[w],w))
winner=min((c for c in candidates if c['width']==width),key=lambda c:(-c['validation']['mAP'],c['seed']))
selection=dict(winner=winner,family_validation_means=means,mode='mlp')
if control['mAP']>=means[width]:selection.update(mode='frozen')
(REPORT/'selection.json').write_text(json.dumps(selection,indent=2))
(REPORT/'candidates.json').write_text(json.dumps(candidates,indent=2))
print('Selection frozen before test:',selection)


## Fase 3. Evaluación y análisis
Congelar la selección antes de test. Evaluar E2 y todos los candidatos en ambas galerías por separado; nunca seleccionar con PetTrace. Diferencias 512−256 emparejadas por semilla e intervalos por identidad condicionados a galería/checkpoint fijos. Registrar también resultados negativos. No registrar modelos ni promover aliases en esta ablación.

In [ ]:
from pettrace.models.artifacts import Model
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.experiments.zero_shot import save_publication_input,load_publication_input
client=mlflow.MlflowClient();publications=[]
for candidate in [dict(mode='frozen',seed=45,folder=str(REPORT/'frozen'))]+candidates:
    mode,seed=candidate['mode'],candidate['seed'];folder=Path(candidate['folder']);folder.mkdir(exist_ok=True)
    head=None
    hashes=dict(base_hashes)
    if mode in MODES:
        saved=torch.load(folder/'head.pt',weights_only=True,map_location='cpu')
        head=triplet.Head(saved['config']['hidden'],dimension=saved['dimension'])
        head.load_state_dict(saved['state_dict']);hashes['head.pt']=sha256(folder/'head.pt')
    encoder,processor=build('frozen' if mode=='raw' else mode,'cuda',Config(seed=seed),head=head)
    if mode=='mlp_lora':
        lora.load_adapter(encoder,folder/'adapter.pt');hashes['adapter.pt']=sha256(folder/'adapter.pt')
    pipeline=Pipeline(encoder,processor,None if mode=='raw' else base.pipeline.detector)
    golden=pipeline.embed(golden_images());assert compare(golden,pipeline.embed(golden_images()))[0]
    np.save(folder/'golden.npy',golden)
    model=Model(f'pettrace-dino://{REPORT.name}/{folder.name}',str(folder),pipeline,f'DINO-{folder.name}','',hashes,True,fingerprint(golden))
    for domain,ds in (('public',benchmarks.public),('private',benchmarks.private)):
        target=folder/domain;target.mkdir(exist_ok=True)
        if (target/'evaluation-input.json').exists():
            ev=load_publication_input(target);ev.dataset=ds
        else:
            ev=evaluate(ds,pipeline,zero_shot=True);save(ev,model,str(target));save_publication_input(ev,target)
            (target/'runtime.json').write_text(json.dumps(dict(**SOURCE,execution_id=REPORT.name,backend='torch_cuda',model_sha256=hashes),indent=2))
        run=publish_evaluation(ev,model,target,SOURCE['source_sha256'],experiment='DINO-'+mode,expected_model_uri=model.uri,expected_model_sha256=hashes,expected_dataset_version=ds.version,expected_protocol='zero-shot')
        assert client.get_run(run).info.status=='FINISHED'
        receipt=dict(mode=mode,width=candidate.get('width'),seed=seed,domain=domain,run_id=run,published=True)
        (target/'publication.json').write_text(json.dumps(receipt,indent=2));client.log_artifact(run,str(target/'publication.json'),'report')
        client.log_artifact(run,str(REPORT/'selection.json'),'study')
        publications.append(receipt)
    del pipeline,encoder;gc.collect();torch.cuda.empty_cache()
assert len(publications)==14
(REPORT/'publication.json').write_text(json.dumps(publications,indent=2))
from pettrace.evaluation.paired import paired_comparison
analysis={}
for domain in ('public','private'):
    def read(folder):
        return json.loads((folder/domain/'evaluation-input.json').read_text())['result']
    baseline=read(REPORT/'frozen')
    groups={w:[read(REPORT/f'mlp-h{w}-seed{s}') for s in SEEDS] for w in WIDTHS}
    analysis[domain]=dict(
        by_width={w:dict(metrics={k:dict(mean=float(np.mean([e['test']['global'][k] for e in group])),sd_seeds=float(np.std([e['test']['global'][k] for e in group],ddof=1))) for k in ('rank1','rank5','mAP')},paired_vs_E2={str(s):paired_comparison(baseline,e) for s,e in zip(SEEDS,group)}) for w,group in groups.items()},
        wider_vs_narrow={str(s):paired_comparison(a,b) for s,a,b in zip(SEEDS,groups[256],groups[512])})
(REPORT/'comparison.json').write_text(json.dumps(analysis,indent=2))
for receipt in publications:client.log_artifact(receipt['run_id'],str(REPORT/'comparison.json'),'study')
assert all(client.get_run(p['run_id']).info.status=='FINISHED' for p in publications)
print('COMPLETE: six training runs and fourteen evaluations published. No registration or alias changes.')
print(json.dumps(analysis,indent=2))
